## Root Cause Analysis - Exp3

* Goals
  * Root Cause Analysis on individual RAG results
  * Provide actionable suggestions
* About Exp3
  * No need to use agentic AI

In [1]:
%load_ext autoreload
%autoreload 2

import os
from sqlalchemy import make_url
import pandas as pd

from utils_root_cause import *

from langchain_groq import ChatGroq
from typing import TypedDict, Literal, Optional
from langgraph.graph import StateGraph, START, END
import json

import warnings
warnings.filterwarnings('ignore')


exp_llm = ChatGroq(
    groq_api_key=os.environ["GROQ_TOKEN"],
    model_name="openai/gpt-oss-20b", 
    temperature=0.78
)

DATABASE_URL_PUBLIC = os.getenv("DATABASE_URL_PUBLIC_RAG_DR")
DATABASE_URL = DATABASE_URL_PUBLIC.replace("postgres://", "postgresql://")
db_url = make_url(DATABASE_URL)

In [3]:
auto_eval_output = get_auto_eval_output(db_url)
print(auto_eval_output.shape)

display(auto_eval_output.head(n=2))

left_auto_eval_output = auto_eval_output.iloc[0:30].head(n=5)
right_auto_eval_output = auto_eval_output.iloc[30:60].head(n=5)
print(left_auto_eval_output.shape, right_auto_eval_output.shape)

(60, 16)


,config_hash,dataset,embedding_model,top_n_retrieval,semantic_weight,answer_gen_llm,query,context,retrieved_content,same_context,retrieval_quality_score,rq_reasoning,referenced_answer,ai_answer,answer_quality_score,aq_reasoning
0,bcfe2f64d957b6c94a8cb20b52464cb9,FIQA Data,BAAI/bge-small-en-v1.5,1,0.5,llama-3.1-8b-instant,How to deposit a cheque issued to an associate...,Just have the associate sign the back and then...,Just have the associate sign the back and then...,true,3,The retrieved content reproduces the entire co...,Have the check reissued to the proper payee.Ju...,To deposit a cheque issued to an associate in ...,2,The AI’s answer correctly identifies that the ...
1,bcfe2f64d957b6c94a8cb20b52464cb9,FIQA Data,BAAI/bge-small-en-v1.5,1,0.5,llama-3.1-8b-instant,Can I send a money order from USPS as a business?,Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...,true,3,The retrieved content matches the context exac...,Sure you can. You can fill in whatever you wa...,"Yes, you can fill in your business name and ad...",2,The AI’s answer correctly states that a busine...


(5, 16) (5, 16)


In [7]:
_llm_sem = asyncio.Semaphore(4)

async def run_rca(record):
    query = record['query']

    async with _llm_sem:
        rq_score_after_review, aq_score_after_review, query_quality = await asyncio.gather(
            review_sr_async(exp_llm, record['rq_reasoning']),
            review_sr_async(exp_llm, record['aq_reasoning']),
            review_query_quality_async(exp_llm, query)
        )

    root_cause_analysis = []

    # ------------- Review AI-as-Judge ------------- #
    if rq_score_after_review != record['retrieval_quality_score']:
        root_cause_analysis.append({'auto eval score issue': f'score vs reasoning mismatch, \
                                    changed retrieval_quality_score from {record["retrieval_quality_score"]} to {rq_score_after_review}'})
    if aq_score_after_review != record['answer_quality_score']:
        root_cause_analysis.append({'auto eval score issue': f'score vs reasoning mismatch, \
                                    changed answer_quality_score from {record["answer_quality_score"]} to {aq_score_after_review}'})

    record['new_retrieval_quality_score'] = rq_score_after_review
    record['new_answer_quality_score'] = aq_score_after_review

    # ------------- Review References ------------- #
    needs_re_eval = 0
    rc_alignment_score, rc_alignment_reason = None, None
    ac_alignment_score, ac_alignment_reason = None, None

    if rq_score_after_review == -1:
        root_cause_analysis.append({'Please review referenced content':
                                            'referenced content is much less relevant to the query than retrieved content'})
        needs_re_eval = 1
    if aq_score_after_review == -1:
        root_cause_analysis.append({"Please review referenced answer":
                                            "referenced answer is much less relevant to the query than AI's answer"})
        needs_re_eval = 1

    if rq_score_after_review >= 2 and aq_score_after_review in [0, 1]:  # good retrieval, bad answer
        referenced_answer_context_alignment = await review_ta_async(exp_llm, query,
                                                                    record['referenced_answer'], record['context'])
        rc_alignment_score = referenced_answer_context_alignment.score
        rc_alignment_reason = referenced_answer_context_alignment.reasoning
        if rc_alignment_score == 0:
            root_cause_analysis.append({'Please review referenced answer':
                                        'referenced answer has critical information misaligned with the referenced content'})
            needs_re_eval = 1
            
    if rq_score_after_review in [0, 1] and aq_score_after_review >= 2:  # good answer, bad retrieval
        ai_answer_context_alignment = await review_ta_async(exp_llm, query,
                                                                    record['ai_answer'], record['context'])
        ac_alignment_score = ai_answer_context_alignment.score
        ac_alignment_reason = ai_answer_context_alignment.reasoning
        if ac_alignment_score == 0:
            root_cause_analysis.append({"Please review referenced content":
                                        "AI's answer got a high score but retrieval score is low, \
                                        please check whether need to add or merge retrieved content into the referenced content."})
            needs_re_eval = 1
    
    record['needs_re_eval'] = needs_re_eval
    record['referenced_answer_context_alignment'] = rc_alignment_score
    record['referenced_answer_context_alignment_reason'] = rc_alignment_reason
    record['ai_answer_context_alignment'] = ac_alignment_score
    record['ai_answer_context_alignment_reason'] = ac_alignment_reason

    # ------------- Review Query Quality ------------- #
    if query_quality == 'ambiguous':
        query_variants = await expand_query_async(exp_llm, query)
        root_cause_analysis.append({'Please review query quality': query_variants})

    record['query_quality'] = query_quality
    record['root_cause_analysis'] = root_cause_analysis
    return record

In [8]:
sem = asyncio.Semaphore(2)  # outer per-record concurrency

async def throttled_run_rca(record):
    async with sem:
        return await run_rca(record)

left_records = left_auto_eval_output.to_dict(orient='records')
right_records = right_auto_eval_output.to_dict(orient='records')

all_results = await asyncio.gather(
    *[throttled_run_rca(r) for r in left_records],
    *[throttled_run_rca(r) for r in right_records]
)

n_left = len(left_records)
left_results = pd.DataFrame(all_results[:n_left])
right_results = pd.DataFrame(all_results[n_left:])

print(left_results.shape, right_results.shape)
display(left_results.head(2))

(5, 25) (5, 25)


,config_hash,dataset,embedding_model,top_n_retrieval,semantic_weight,answer_gen_llm,query,context,retrieved_content,same_context,...,aq_reasoning,new_retrieval_quality_score,new_answer_quality_score,needs_re_eval,referenced_answer_context_alignment,referenced_answer_context_alignment_reason,ai_answer_context_alignment,ai_answer_context_alignment_reason,query_quality,root_cause_analysis
0,bcfe2f64d957b6c94a8cb20b52464cb9,FIQA Data,BAAI/bge-small-en-v1.5,1,0.5,llama-3.1-8b-instant,How to deposit a cheque issued to an associate...,Just have the associate sign the back and then...,Just have the associate sign the back and then...,true,...,The AI’s answer correctly identifies that the ...,3,2,0,None,None,None,None,clear,[{'auto eval score issue': 'score vs reasoning...
1,bcfe2f64d957b6c94a8cb20b52464cb9,FIQA Data,BAAI/bge-small-en-v1.5,1,0.5,llama-3.1-8b-instant,Can I send a money order from USPS as a business?,Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...,true,...,The AI’s answer correctly states that a busine...,3,2,0,None,None,None,None,clear,[{'auto eval score issue': 'score vs reasoning...
